# 실습 2-3. 공정 실험: 조건을 바꿔 사출기를 돌려 보기

강사 화면의 가상 공장은 **공정 모델**이 계산한 결과를 3D 로 보여 줍니다. 이 노트북은 그 공정 모델을 그대로 불러와서, 내 PC 에서 사출기를 직접 돌려 봅니다.

- 조건(금형온도 · 보압 · 냉각 시간 …)을 **내가 정하고**, 샷을 만들고, 불량률을 확인합니다.
- `# 내 값` 표시가 있는 줄을 고친 뒤 그 셀부터 아래로 다시 실행합니다.
- 실행: 셀을 선택하고 `Shift` + `Enter`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists():      # 저장소 최상위 폴더 찾기
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from common import process_model as pm
from common.plot_style import use_korean_font
use_korean_font()

DATA = ROOT / "data" / "session2"
OUT = DATA / "work"                        # 내 산출물이 저장되는 곳
OUT.mkdir(exist_ok=True)
LIVE = DATA / "live"                       # 강사 PC 에서 받은 데이터 (./scripts/factory fetch <주소>)
pd.set_option("display.width", 160)

KO = dict(mold_temp_c="금형온도 (℃)", melt_temp_c="수지온도 (℃)", inj_pressure_mpa="사출압 (MPa)",
          hold_pressure_mpa="보압 (MPa)", hold_time_s="보압 시간 (s)", cooling_time_s="냉각 시간 (s)")
print("기준 조건")
for k, v in pm.INJ_NOMINAL.items():
    print(f"  {KO[k]:<14} {v}")

## 1. 기준 조건으로 300샷

`run(조건, 샷 수)` 는 사출기 한 대를 그 조건으로 돌려 샷 기록을 표로 돌려줍니다. 샷마다 값이 조금씩 흔들리는 것(설비 변동)도 들어 있습니다.

In [ ]:
def run(setpoint=None, n=300, seed=1):
    """설정값을 바꾼 사출기로 n 샷을 만들어 표로 돌려줌."""
    m = pm.InjectionMachine(seed=seed)
    m.setpoint.update(setpoint or {})
    df = pd.DataFrame([m.next_shot().as_dict() for _ in range(n)])
    df["is_defect"] = (df["defect"] != "none").astype(int)
    return df

base = run()
print(f"샷 {len(base)}개, 불량률 {base['is_defect'].mean():.1%}, 평균 사이클타임 {base['cycle_time_s'].mean():.1f}초")
print(base["defect"].map(pm.DEFECT_KO).value_counts().to_string())
base.head()

## 2. 조건 하나를 바꿔 보기

2회차 2교시에 강사 화면에서 본 "금형 온조기 이상"을 직접 재현합니다. 금형온도 설정을 60 → 54 ℃ 로 낮춥니다.

In [ ]:
MY = dict(mold_temp_c=54.0)          # 내 값: 바꿀 조건과 값. 예) dict(cooling_time_s=24.0), dict(hold_pressure_mpa=50.0)

mine = run(MY)
cmp = pd.DataFrame({"기준 조건": base["defect"].map(pm.DEFECT_KO).value_counts(normalize=True),
                    "내 조건": mine["defect"].map(pm.DEFECT_KO).value_counts(normalize=True)}).fillna(0).drop("양품", errors="ignore")
ax = (cmp * 100).plot.bar(figsize=(7, 3.6), rot=0, color=["#8da0cb", "#fc8d62"])
ax.set_xlabel(""); ax.set_ylabel("발생 비율 (%)"); ax.set_title(f"조건 변경: {', '.join(f'{KO[k]} {v}' for k, v in MY.items())}")
plt.tight_layout(); plt.show()
print(f"불량률  기준 {base['is_defect'].mean():.1%}  →  내 조건 {mine['is_defect'].mean():.1%}")

## 3. 한 조건을 훑어 보기

조건 하나를 범위 전체로 바꿔 가며 결함별 발생 확률을 그립니다. 어느 값부터 불량이 늘기 시작하는지(**관리 한계**)를 찾습니다.

In [ ]:
def defect_rate(setpoint, n=3000, seed=0):
    """그 설정값에서의 결함별 발생 확률 (샷 변동 포함)."""
    rng = np.random.default_rng(seed)
    sp = dict(pm.INJ_NOMINAL); sp.update(setpoint)
    v = {k: sp[k] + rng.normal(0, pm.INJ_NOISE[k], n) for k in pm.INJ_NOMINAL}
    p = pm.injection_defect_probs(**v)
    out = {pm.DEFECT_KO[k]: float(p[k].mean()) for k in pm.INJ_DEFECTS}
    out["전체"] = float(1 - ((1 - p["flow_mark"]) * (1 - p["sink"]) * (1 - p["short_shot"])).mean())
    return out

KEY = "mold_temp_c"                  # 내 값: 훑을 조건
lo, hi = pm.INJ_RANGE[KEY]
grid = np.linspace(lo, hi, 41)
sw = pd.DataFrame([defect_rate({KEY: g}) for g in grid], index=grid) * 100
ax = sw.plot(figsize=(8, 3.8), lw=2)
ax.axvline(pm.INJ_NOMINAL[KEY], color="gray", ls=":"); ax.axhline(3, color="red", ls="--", lw=1)
ax.set_xlabel(KO[KEY]); ax.set_ylabel("발생 확률 (%)"); ax.set_title("점선: 기준 값, 빨간 선: 불량률 3%")
plt.tight_layout(); plt.show()
ok = sw.index[sw["전체"] <= 3.0]
print(f"불량률 3% 이하인 범위: {ok.min():.1f} ~ {ok.max():.1f}  (기준 {pm.INJ_NOMINAL[KEY]})")

## 4. 두 조건을 함께 보기

조건 두 개를 격자로 바꿔 불량률 지도를 그립니다. 밝은 영역이 안전한 조건입니다.

In [ ]:
KX, KY = "cooling_time_s", "hold_pressure_mpa"       # 내 값: 가로 · 세로 조건
gx, gy = np.linspace(*pm.INJ_RANGE[KX], 25), np.linspace(*pm.INJ_RANGE[KY], 25)
Z = np.array([[defect_rate({KX: x, KY: y}, n=600)["전체"] for x in gx] for y in gy]) * 100
fig, ax = plt.subplots(figsize=(7, 4.6))
im = ax.pcolormesh(gx, gy, Z, cmap="RdYlGn_r", vmin=0, vmax=30, shading="auto")
ax.contour(gx, gy, Z, levels=[3], colors="k", linewidths=1.5)
ax.plot(pm.INJ_NOMINAL[KX], pm.INJ_NOMINAL[KY], "k*", ms=14)
ax.set_xlabel(KO[KX]); ax.set_ylabel(KO[KY]); ax.set_title("불량률 (%) · 검은 선: 3% 경계 · 별: 기준 조건")
fig.colorbar(im); plt.tight_layout(); plt.show()

## 5. 미션: 사이클타임을 줄여 생산량 늘리기

생산 관리자가 "사이클타임을 줄여 달라"고 합니다. 냉각 시간 · 보압 시간을 줄이면 사이클타임은 짧아지지만 수축 불량이 늘어납니다.

- 목표: **불량률 3% 이하**를 지키면서 **시간당 양품 수**를 가장 크게
- `MY_BEST` 를 바꿔 가며 아래 셀을 다시 실행합니다.

In [ ]:
MY_BEST = dict(cooling_time_s=28.0, hold_time_s=8.0)      # 내 값

def evaluate(sp, n=2000):
    df = run(sp, n=n, seed=7)
    rate, ct = df["is_defect"].mean(), df["cycle_time_s"].mean()
    return dict(불량률=rate, 사이클타임_초=ct, 시간당_양품=3600 / ct * (1 - rate))

rows = {"기준 조건": evaluate({}), "내 조건": evaluate(MY_BEST)}
res = pd.DataFrame(rows).T
res["판정"] = np.where(res["불량률"] <= 0.03, "합격", "불량률 초과")
display(res.style.format({"불량률": "{:.1%}", "사이클타임_초": "{:.1f}", "시간당_양품": "{:.1f}"}))
pd.DataFrame([dict(MY_BEST, **rows["내 조건"])]).to_csv(OUT / "process_lab_result.csv", index=False)
print("저장:", (OUT / "process_lab_result.csv").relative_to(ROOT))

## 6. 강사 PC 의 가상 공장 데이터 받아 보기

강사 화면의 가상 공장이 방금 만든 기록을 내 PC 로 받아 같은 방식으로 봅니다. 터미널에서 먼저 실행합니다 (주소는 강사 화면의 "수강생 연결" 구역에 있음).

```bash
cd ~/ai_manufac
./scripts/factory fetch <강사 PC 주소>
```

In [ ]:
shots = sorted(LIVE.glob("shots_*.csv"))
if not shots:
    print("받은 파일이 없습니다. 위의 fetch 명령을 먼저 실행하세요. (없어도 다음 실습에는 지장 없음)")
else:
    s = pd.read_csv(shots[-1])
    ins = pd.read_csv(LIVE / shots[-1].name.replace("shots_", "inspection_"))
    print(f"{shots[-1].name}: 샷 {len(s)}행, 검사 {len(ins)}행, 검사 불량률 {(ins['result'] != 'OK').mean():.1%}")
    cond = list(pm.INJ_NOMINAL)
    live_mean = s[cond].apply(pd.to_numeric, errors="coerce").mean()
    print("이 조건을 공정 모델에 넣었을 때의 기대 불량률:", f"{defect_rate(live_mean.to_dict())['전체']:.1%}")
    display(pd.DataFrame({"가상 공장 평균": live_mean.round(1), "기준 조건": pd.Series(pm.INJ_NOMINAL)}).rename(index=KO))

## 정리

- 조건을 바꾸면 **어떤 결함이** 늘어나는지가 조건마다 다릅니다 (금형온도 ↓ → 흐름자국, 냉각 시간 ↓ → 수축).
- 불량은 어느 값부터 갑자기 늘어납니다. 그 경계가 **관리 한계**이고, 3회차의 조건 추천이 찾는 것도 이 경계입니다.
- 사이클타임과 불량률은 서로 맞바꾸는 관계입니다. 어디까지 줄일지는 데이터로 정합니다.